# W8D2 — RAG Evaluation and Optimization

## Objective
Build and evaluate a Retrieval-Augmented Generation (RAG) pipeline using LangChain, ChromaDB, and Ragas.

## Tasks
1. Load the existing dataset.
2. Split the dataset into chunks and index them in ChromaDB.
3. Generate 10 question-answer pairs.
4. Evaluate faithfulness, answer relevancy, context precision, and context recall.
5. Improve the lowest-scoring metric by changing chunk size or retrieval k.
6. Re-evaluate and compare baseline and optimized results.
7. Document findings and complete the Git workflow.

In [1]:
from pathlib import Path

dataset_path = Path("W7D4_output.txt")

if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found: {dataset_path.resolve()}")

text = dataset_path.read_text(encoding="utf-8")

print("Dataset loaded successfully!")
print("Characters:", len(text))
print("Words:", len(text.split()))
print("\nPreview:\n")
print(text[:500])

Dataset loaded successfully!
Characters: 7636
Words: 1140

Preview:


--- Prompt 1 ---
Question: What is artificial intelligence?
Answer:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. Learning: AI systems can learn from data and improve their performance over time.
2. Problem-solving: AI systems can analyze problems and find solutions.
3. Perception: AI systems can interpret and understand data from sensors and other sources.
4. Reasoning: AI systems can dr


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(text)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0][:300])

Total chunks: 22

First chunk preview:

--- Prompt 1 ---
Question: What is artificial intelligence?
Answer:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="w8d2_rag_evaluation"
)

print("Embeddings created successfully!")
print("Chunks indexed:", vectorstore._collection.count())

c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 886.29it/s]


Embeddings created successfully!
Chunks indexed: 22


In [4]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

test_docs = retriever.invoke(
    "What is artificial intelligence?"
)

print("Retriever working:", len(test_docs) > 0)
print("Retrieved documents:", len(test_docs))
print("\nFirst retrieved result:\n")
print(test_docs[0].page_content[:400])

Retriever working: True
Retrieved documents: 3

First retrieved result:

--- Prompt 1 ---
Question: What is artificial intelligence?
Answer:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:


In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
If the context does not contain the answer, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

def rag_answer(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = rag_prompt.format(context=context, question=question)
    response = llm.invoke(prompt)

    return {
        "question": question,
        "answer": response.content,
        "contexts": [doc.page_content for doc in docs]
    }

test_result = rag_answer("What is artificial intelligence?")
print(test_result["answer"])

Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

In simple terms, AI is like a supercomputer that can think and learn like a human, but with its own unique strengths and weaknesses.

Example: Imagine you're a self-driving car. It uses AI to learn from data, recognize patterns, and make decisions about how to navigate roads, avoiding obstacles, and following traffic rules. The car's "brain" is essentially a sophisticated AI system that enables it to perform tasks that would be challenging or impossible for humans to do alone.

AI is not a single technology, but rather a broad field that encompasses various techniques and methods, such as:

1. Learning: AI systems can learn from data and improve their performance over time.
2. Problem-solving: AI systems can analyze problems and find solutions.
3. Perception: AI systems can interpret and understand data from sensors and other sources.
4.

In [6]:
evaluation_data = [
    {
        "question": "What is artificial intelligence?",
        "ground_truth": "Artificial intelligence is the development of computer systems that perform tasks requiring human-like intelligence."
    },
    {
        "question": "What are four tasks AI systems can perform?",
        "ground_truth": "AI systems can learn from data, solve problems, interpret sensor data, and reason to draw conclusions and make decisions."
    },
    {
        "question": "What is machine learning?",
        "ground_truth": "Machine learning is a subset of AI that develops algorithms capable of learning from data."
    },
    {
        "question": "What is deep learning?",
        "ground_truth": "Deep learning is a type of machine learning that uses neural networks to analyze complex data."
    },
    {
        "question": "What is natural language processing?",
        "ground_truth": "Natural language processing is concerned with processing and understanding human language."
    },
    {
        "question": "How can AI be used in robotics?",
        "ground_truth": "AI can be used to control and interact with physical devices."
    },
    {
        "question": "How does an AI system improve its performance?",
        "ground_truth": "An AI system can learn from data and improve its performance over time."
    },
    {
        "question": "How can AI help a self-driving car?",
        "ground_truth": "AI can help a self-driving car recognize patterns, navigate roads, avoid obstacles, and follow traffic rules."
    },
    {
        "question": "How does AI solve problems?",
        "ground_truth": "AI systems analyze problems and identify possible solutions."
    },
    {
        "question": "How does AI use reasoning?",
        "ground_truth": "AI systems use reasoning to draw conclusions and make decisions based on available information."
    }
]

print("Evaluation questions prepared:", len(evaluation_data))

Evaluation questions prepared: 10


In [7]:
import json
from pathlib import Path

evaluation_results = []

for i, item in enumerate(evaluation_data, start=1):
    result = rag_answer(item["question"])
    result["ground_truth"] = item["ground_truth"]
    evaluation_results.append(result)
    print(f"Completed {i}/10")

results_path = Path("W8D2_evaluation_results.json")
results_path.write_text(
    json.dumps(evaluation_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nAll 10 answers generated and saved!")
print("Saved to:", results_path.resolve())

Completed 1/10
Completed 2/10
Completed 3/10
Completed 4/10
Completed 5/10
Completed 6/10
Completed 7/10
Completed 8/10
Completed 9/10
Completed 10/10

All 10 answers generated and saved!
Saved to: C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks\W8D2_evaluation_results.json


In [8]:
from datasets import Dataset
import json
from pathlib import Path

results_path = Path("W8D2_evaluation_results.json")
evaluation_results = json.loads(
    results_path.read_text(encoding="utf-8")
)

ragas_dataset = Dataset.from_list([
    {
        "question": item["question"],
        "answer": item["answer"],
        "contexts": item["contexts"],
        "ground_truth": item["ground_truth"]
    }
    for item in evaluation_results
])

print("Ragas dataset prepared:", len(ragas_dataset))
print("Columns:", ragas_dataset.column_names)

Ragas dataset prepared: 10
Columns: ['question', 'answer', 'contexts', 'ground_truth']


In [9]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

print("Metrics imported successfully:")
print("1. Faithfulness")
print("2. Answer relevancy")
print("3. Context precision")
print("4. Context recall")

Metrics imported successfully:
1. Faithfulness
2. Answer relevancy
3. Context precision
4. Context recall


In [10]:
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(
    ChatOllama(
        model="qwen2.5:3b",
        temperature=0,
        format="json",
        num_predict=256,
        timeout=120
    )
)

evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

faithfulness.llm = evaluator_llm
answer_relevancy.llm = evaluator_llm
answer_relevancy.embeddings = evaluator_embeddings
context_precision.llm = evaluator_llm
context_recall.llm = evaluator_llm

print("Local evaluator configured.")
print("LLM: qwen2.5:3b")
print("Embeddings: all-MiniLM-L6-v2")
print("Metrics configured: 4")

Local evaluator configured.
LLM: qwen2.5:3b
Embeddings: all-MiniLM-L6-v2
Metrics configured: 4


In [11]:
test_question = ragas_dataset[0]["question"]
test_answer = ragas_dataset[0]["answer"]

test_embedding = evaluator_embeddings.embed_query(test_question)

print("Question:", test_question)
print("Answer available:", bool(test_answer))
print("Embedding dimensions:", len(test_embedding))

Question: What is artificial intelligence?
Answer available: True
Embedding dimensions: 384


In [13]:
from ragas.metrics import answer_relevancy

print([
    name for name in dir(answer_relevancy)
    if "prompt" in name.lower()
])

['_get_prompts', 'adapt_prompts', 'get_prompts', 'load_prompts', 'save_prompts', 'set_prompts']


In [14]:
prompts = answer_relevancy.get_prompts()

print("Available prompt names:", list(prompts.keys()))

for name, prompt in prompts.items():
    print(f"\n--- {name} ---")
    print(prompt)

Available prompt names: ['response_relevance_prompt']

--- response_relevance_prompt ---
ResponseRelevancePrompt(
  "name": "response_relevance_prompt",
  "instruction": "Generate a question for the given answer and Identify if answer is noncommittal. Give noncommittal as 1 if the answer is noncommittal and 0 if the answer is committal. A noncommittal answer is one that is evasive, vague, or ambiguous. For example, \"I don't know\" or \"I'm not sure\" are noncommittal answers",
  "examples": [
    [
      {
        "response": "Albert Einstein was born in Germany."
      },
      {
        "question": "Where was Albert Einstein born?",
        "noncommittal": 0
      }
    ],
    [
      {
        "response": "I don't know about the  groundbreaking feature of the smartphone invented in 2023 as am unaware of information beyond 2022. "
      },
      {
        "question": "What was the groundbreaking feature of the smartphone invented in 2023?",
        "noncommittal": 1
      }
    ]
  

In [15]:
relevance_prompt = answer_relevancy.get_prompts()["response_relevance_prompt"]

relevance_data = relevance_prompt.input_model(
    response=ragas_dataset[0]["answer"]
)

print(relevance_data)

response='Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:\n\nIn simple terms, AI is like a supercomputer that can think and learn like a human, but with its own unique strengths and weaknesses.\n\nExample: Imagine you\'re a self-driving car. It uses AI to learn from data, recognize patterns, and make decisions about how to navigate roads, avoiding obstacles, and following traffic rules. The car\'s "brain" is essentially a sophisticated AI system that enables it to perform tasks that would be challenging or impossible for humans to do alone.\n\nAI is not a single technology, but rather a broad field that encompasses various techniques and methods, such as:\n\n1. Learning: AI systems can learn from data and improve their performance over time.\n2. Problem-solving: AI systems can analyze problems and find solutions.\n3. Perception: AI systems can interpret and understand data from sensors

In [16]:
relevance_result = await relevance_prompt.generate(
    llm=evaluator_llm,
    data=relevance_data,
    retries_left=0
)

print(relevance_result)

question='What are the key techniques used in Artificial Intelligence (AI)?' noncommittal=0


In [17]:
from ragas.metrics import faithfulness

faithfulness_data = faithfulness.nli_statements_prompt.input_model(
    context="\n".join(ragas_dataset[0]["contexts"]),
    statements=[
        "AI systems can learn from data and improve their performance over time."
    ]
)

faithfulness_check = await faithfulness.nli_statements_prompt.generate(
    llm=evaluator_llm,
    data=faithfulness_data,
    retries_left=0
)

print(faithfulness_check)

statements=[StatementFaithfulnessAnswer(statement='AI systems can learn from data and improve their performance over time.', reason='This statement is directly stated in the context provided. The context explicitly mentions that AI systems can learn from data and improve their performance over time.', verdict=1)]


In [18]:
async def score_faithfulness(question, answer, contexts):
    from ragas.metrics import faithfulness

    statements_data = faithfulness.nli_statements_prompt.input_model(
        context="\n".join(contexts),
        statements=[answer]
    )

    result = await faithfulness.nli_statements_prompt.generate(
        llm=evaluator_llm,
        data=statements_data,
        retries_left=0
    )

    verdicts = result.statements
    if not verdicts:
        return None

    return sum(v.verdict for v in verdicts) / len(verdicts)


print("Faithfulness scoring function created.")

Faithfulness scoring function created.


In [20]:
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(
    ChatOllama(
        model="llama3.2:3b",
        temperature=0,
        format="json",
        num_predict=512,
        timeout=120
    )
)

faithfulness.llm = evaluator_llm
answer_relevancy.llm = evaluator_llm
context_precision.llm = evaluator_llm
context_recall.llm = evaluator_llm

print("Evaluator switched to llama3.2:3b with JSON mode.")

Evaluator switched to llama3.2:3b with JSON mode.


In [21]:
from ragas.metrics import faithfulness

faithfulness_data = faithfulness.nli_statements_prompt.input_model(
    context="Artificial intelligence systems can learn from data.",
    statements=["AI systems can learn from data."]
)

faithfulness_check = await faithfulness.nli_statements_prompt.generate(
    llm=evaluator_llm,
    data=faithfulness_data,
    retries_left=0
)

print(faithfulness_check)

statements=[StatementFaithfulnessAnswer(statement='AI systems can learn from data.', reason='The context explicitly states that AI systems can learn from data, making this statement directly inferable.', verdict=1)]


In [22]:
import json
from pathlib import Path

faithfulness_scores = []

for i, item in enumerate(evaluation_results, start=1):
    data = faithfulness.nli_statements_prompt.input_model(
        context="\n".join(item["contexts"]),
        statements=[item["answer"]]
    )

    result = await faithfulness.nli_statements_prompt.generate(
        llm=evaluator_llm,
        data=data,
        retries_left=0
    )

    verdicts = result.statements
    score = (
        sum(v.verdict for v in verdicts) / len(verdicts)
        if verdicts else None
    )

    faithfulness_scores.append({
        "question": item["question"],
        "faithfulness": score
    })

    print(f"{i}/10 completed — faithfulness: {score}")

Path("W8D2_faithfulness_scores.json").write_text(
    json.dumps(faithfulness_scores, indent=2),
    encoding="utf-8"
)

print("\nFaithfulness results saved.")

1/10 completed — faithfulness: 1.0
2/10 completed — faithfulness: 1.0
3/10 completed — faithfulness: 1.0
4/10 completed — faithfulness: 1.0
5/10 completed — faithfulness: 1.0
6/10 completed — faithfulness: 1.0
7/10 completed — faithfulness: 1.0
8/10 completed — faithfulness: 0.7142857142857143
9/10 completed — faithfulness: 1.0
10/10 completed — faithfulness: 1.0

Faithfulness results saved.


In [23]:
import json
from pathlib import Path

faithfulness_average = sum(
    item["faithfulness"] for item in faithfulness_scores
) / len(faithfulness_scores)

baseline_summary = {
    "faithfulness_average": faithfulness_average,
    "faithfulness_scores": faithfulness_scores,
    "scoring_note": (
        "Preliminary scoring: each complete answer was treated as one "
        "statement. This is not a rigorous claim-level Ragas score."
    )
}

Path("W8D2_baseline_summary.json").write_text(
    json.dumps(baseline_summary, indent=2),
    encoding="utf-8"
)

print("Average faithfulness:", round(faithfulness_average, 4))
print("Baseline summary saved.")

Average faithfulness: 0.9714
Baseline summary saved.


In [24]:
from ragas.metrics import answer_relevancy

relevance_prompt = answer_relevancy.get_prompts()["response_relevance_prompt"]

async def score_answer_relevancy(answer):
    relevance_data = relevance_prompt.input_model(
        response=answer
    )

    result = await relevance_prompt.generate(
        llm=evaluator_llm,
        data=relevance_data,
        retries_left=0
    )

    return {
        "generated_question": result.question,
        "noncommittal": result.noncommittal
    }

print("Answer-relevancy prompt function created.")

Answer-relevancy prompt function created.


In [25]:
import json
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

relevancy_scores = []

for i, item in enumerate(evaluation_results, start=1):
    result = await score_answer_relevancy(item["answer"])

    original_embedding = evaluator_embeddings.embed_query(
        item["question"]
    )
    generated_embedding = evaluator_embeddings.embed_query(
        result["generated_question"]
    )

    similarity = float(cosine_similarity(
        [original_embedding],
        [generated_embedding]
    )[0][0])

    score = 0.0 if result["noncommittal"] == 1 else max(0.0, similarity)

    relevancy_scores.append({
        "question": item["question"],
        "generated_question": result["generated_question"],
        "noncommittal": result["noncommittal"],
        "answer_relevancy": score
    })

    print(f"{i}/10 completed — relevancy: {score:.4f}")

Path("W8D2_answer_relevancy_scores.json").write_text(
    json.dumps(relevancy_scores, indent=2),
    encoding="utf-8"
)

print("\nAnswer-relevancy results saved.")

1/10 completed — relevancy: 0.8474
2/10 completed — relevancy: 0.9817
3/10 completed — relevancy: 0.7944
4/10 completed — relevancy: 0.7291
5/10 completed — relevancy: 0.5371
6/10 completed — relevancy: 0.8878
7/10 completed — relevancy: 1.0000
8/10 completed — relevancy: 0.9429
9/10 completed — relevancy: 1.0000
10/10 completed — relevancy: 0.9034

Answer-relevancy results saved.


In [26]:
import json
from pathlib import Path

relevancy_average = sum(
    item["answer_relevancy"] for item in relevancy_scores
) / len(relevancy_scores)

baseline_summary["answer_relevancy_average"] = relevancy_average
baseline_summary["answer_relevancy_scores"] = relevancy_scores

Path("W8D2_baseline_summary.json").write_text(
    json.dumps(baseline_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Average answer relevancy:", round(relevancy_average, 4))
print("Lowest question:", min(
    relevancy_scores,
    key=lambda x: x["answer_relevancy"]
)["question"])
print("Baseline summary updated.")

Average answer relevancy: 0.8624
Lowest question: What is natural language processing?
Baseline summary updated.


In [27]:
from ragas.metrics import context_precision

print("Available prompts:", list(context_precision.get_prompts().keys()))

for name, prompt in context_precision.get_prompts().items():
    print(f"\n--- {name} ---")
    print(prompt)
    

Available prompts: ['context_precision_prompt']

--- context_precision_prompt ---
ContextPrecisionPrompt(
  "name": "context_precision_prompt",
  "instruction": "Given question, answer and context verify if the context was useful in arriving at the given answer. Give verdict as \"1\" if useful and \"0\" if not with json output.",
  "examples": [
    [
      {
        "question": "What can you tell me about Albert Einstein?",
        "context": "Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist, widely held to be one of the greatest and most influential scientists of all time. Best known for developing the theory of relativity, he also made important contributions to quantum mechanics, and was thus a central figure in the revolutionary reshaping of the scientific understanding of nature that modern physics accomplished in the first decades of the twentieth century. His mass–energy equivalence formula E = mc2, which arises from relativity theory, has

In [28]:

from ragas.metrics import context_precision

precision_prompt = context_precision.get_prompts()["context_precision_prompt"]

async def score_context_precision(question, answer, contexts):
    verdicts = []

    for context in contexts:
        precision_data = precision_prompt.input_model(
            question=question,
            answer=answer,
            context=context
        )

        result = await precision_prompt.generate(
            llm=evaluator_llm,
            data=precision_data,
            retries_left=0
        )

        verdicts.append(int(result.verdict))

    if not verdicts:
        return 0.0

    return sum(verdicts) / len(verdicts)

print("Context-precision scoring function created.")

Context-precision scoring function created.


In [30]:
import json

async def score_context_precision(question, answer, contexts):
    verdicts = []

    for context in contexts:
        prompt = f"""
Judge whether the context is useful for answering the question.

Question: {question}
Answer: {answer}
Context: {context}

Return JSON only:
{{"verdict": 1, "reason": "brief explanation"}}

Use verdict 1 if useful, otherwise 0.
"""
        response = await evaluator_llm.llm.ainvoke(prompt)
        data = json.loads(response.content)
        verdicts.append(int(data["verdict"]))

    return sum(verdicts) / len(verdicts) if verdicts else 0.0

print("JSON-based context-precision function ready.")

JSON-based context-precision function ready.


In [33]:
import json

async def score_context_precision(question, answer, contexts):
    verdicts = []

    for context in contexts:
        prompt = f"""
Judge whether the context is useful for answering the question.

Question: {question}
Answer: {answer}
Context: {context}

Return JSON only:
{{"verdict": 1, "reason": "brief explanation"}}

Use verdict 1 if useful, otherwise 0.
"""
        response = await json_evaluator.langchain_llm.ainvoke(prompt)
        data = json.loads(response.content)
        verdicts.append(int(data["verdict"]))

    return sum(verdicts) / len(verdicts) if verdicts else 0.0

print("Context-precision function updated.")

Context-precision function updated.


In [35]:
print("Available evaluator variables:")
print("evaluator_llm:", type(evaluator_llm).__name__)
print("llm:", type(llm).__name__)

Available evaluator variables:
evaluator_llm: LangchainLLMWrapper
llm: ChatOllama


In [36]:
import json

async def score_context_precision(question, answer, contexts):
    verdicts = []

    for context in contexts:
        prompt = f"""
Judge whether the context is useful for answering the question.

Question: {question}
Answer: {answer}
Context: {context}

Return JSON only:
{{"verdict": 1, "reason": "brief explanation"}}

Use verdict 1 if useful, otherwise 0.
"""
        response = await llm.ainvoke(prompt, format="json")
        data = json.loads(response.content)
        verdicts.append(int(data["verdict"]))

    return sum(verdicts) / len(verdicts) if verdicts else 0.0

precision_test = await score_context_precision(
    question=evaluation_results[0]["question"],
    answer=evaluation_results[0]["answer"],
    contexts=evaluation_results[0]["contexts"]
)

print("Context precision estimate:", round(precision_test, 4))

Context precision estimate: 1.0


In [37]:
import json
from pathlib import Path

context_precision_scores = []

for i, item in enumerate(evaluation_results, start=1):
    score = await score_context_precision(
        question=item["question"],
        answer=item["answer"],
        contexts=item["contexts"]
    )

    context_precision_scores.append({
        "question": item["question"],
        "context_precision": score
    })

    print(f"{i}/10 completed — context precision: {score:.4f}")

Path("W8D2_context_precision_scores.json").write_text(
    json.dumps(context_precision_scores, indent=2),
    encoding="utf-8"
)

print("\nContext precision results saved.")

1/10 completed — context precision: 1.0000
2/10 completed — context precision: 0.6667
3/10 completed — context precision: 1.0000
4/10 completed — context precision: 0.6667
5/10 completed — context precision: 0.6667
6/10 completed — context precision: 1.0000
7/10 completed — context precision: 1.0000
8/10 completed — context precision: 0.6667
9/10 completed — context precision: 0.6667
10/10 completed — context precision: 0.6667

Context precision results saved.


In [38]:
import json
from pathlib import Path

context_precision_average = sum(
    item["context_precision"] for item in context_precision_scores
) / len(context_precision_scores)

baseline_summary["context_precision_average"] = context_precision_average
baseline_summary["context_precision_scores"] = context_precision_scores

Path("W8D2_baseline_summary.json").write_text(
    json.dumps(baseline_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Average context precision estimate:", round(context_precision_average, 4))
print("Baseline summary updated.")

Average context precision estimate: 0.8
Baseline summary updated.


In [39]:
import json

async def score_context_recall(question, ground_truth, contexts):
    prompt = f"""
Determine whether the retrieved context contains the information
needed to support the reference answer.

Question: {question}
Reference answer: {ground_truth}

Retrieved context:
{" ".join(contexts)}

Return JSON only:
{{"verdict": 1, "reason": "brief explanation"}}

Use verdict 1 if the context supports the reference answer,
otherwise 0.
"""
    response = await llm.ainvoke(prompt, format="json")
    result = json.loads(response.content)

    return {
        "score": int(result["verdict"]),
        "reason": result.get("reason", "")
    }

print("Context-recall scoring function created.")

Context-recall scoring function created.


In [40]:
recall_test = await score_context_recall(
    question=evaluation_results[0]["question"],
    ground_truth=evaluation_results[0]["ground_truth"],
    contexts=evaluation_results[0]["contexts"]
)

print("Question:", evaluation_results[0]["question"])
print("Context recall estimate:", recall_test["score"])
print("Reason:", recall_test["reason"])

Question: What is artificial intelligence?
Context recall estimate: 1
Reason: brief explanation


In [41]:
import json
from pathlib import Path

context_recall_scores = []

for i, item in enumerate(evaluation_results, start=1):
    result = await score_context_recall(
        question=item["question"],
        ground_truth=item["ground_truth"],
        contexts=item["contexts"]
    )

    context_recall_scores.append({
        "question": item["question"],
        "context_recall": result["score"],
        "reason": result["reason"]
    })

    print(f"{i}/10 completed — context recall: {result['score']}")

context_recall_average = sum(
    item["context_recall"] for item in context_recall_scores
) / len(context_recall_scores)

Path("W8D2_context_recall_scores.json").write_text(
    json.dumps(context_recall_scores, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

baseline_summary["context_recall_average"] = context_recall_average
baseline_summary["context_recall_scores"] = context_recall_scores

Path("W8D2_baseline_summary.json").write_text(
    json.dumps(baseline_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nAverage context recall estimate:", round(context_recall_average, 4))
print("Context-recall results and baseline summary saved.")

1/10 completed — context recall: 1
2/10 completed — context recall: 1
3/10 completed — context recall: 1
4/10 completed — context recall: 1
5/10 completed — context recall: 1
6/10 completed — context recall: 1
7/10 completed — context recall: 1
8/10 completed — context recall: 1
9/10 completed — context recall: 1
10/10 completed — context recall: 1

Average context recall estimate: 1.0
Context-recall results and baseline summary saved.


In [42]:
# Optimization: increase retrieval k from 3 to 4
optimized_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

# Test the optimized retriever
test_docs_optimized = optimized_retriever.invoke(
    "What is artificial intelligence?"
)

print("Baseline retrieval k:", 3)
print("Optimized retrieval k:", 4)
print("Retrieved documents:", len(test_docs_optimized))
print("\nFirst retrieved context:\n")
print(test_docs_optimized[0].page_content[:300])

Baseline retrieval k: 3
Optimized retrieval k: 4
Retrieved documents: 4

First retrieved context:

--- Prompt 1 ---
Question: What is artificial intelligence?
Answer:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:


In [43]:
import json
from pathlib import Path

# Generate answers using the optimized retriever
retriever_before_optimization = retriever
retriever = optimized_retriever

try:
    optimized_results = []

    for i, item in enumerate(evaluation_data, start=1):
        result = rag_answer(item["question"])
        result["ground_truth"] = item["ground_truth"]
        optimized_results.append(result)
        print(f"Completed {i}/10")

finally:
    retriever = retriever_before_optimization

Path("W8D2_optimized_results.json").write_text(
    json.dumps(optimized_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nOptimized answers saved to W8D2_optimized_results.json")
print("Baseline retriever restored.")

Completed 1/10
Completed 2/10
Completed 3/10
Completed 4/10
Completed 5/10
Completed 6/10
Completed 7/10
Completed 8/10
Completed 9/10
Completed 10/10

Optimized answers saved to W8D2_optimized_results.json
Baseline retriever restored.


In [45]:
import json
from pathlib import Path

optimized_results = json.loads(
    Path("W8D2_optimized_results.json").read_text(encoding="utf-8")
)

optimized_faithfulness_scores = []

for i, item in enumerate(optimized_results, start=1):
    score = await score_faithfulness(
        question=item["question"],
        answer=item["answer"],
        contexts=item["contexts"]
    )

    optimized_faithfulness_scores.append({
        "question": item["question"],
        "faithfulness": score
    })

    print(f"{i}/10 completed — faithfulness estimate: {score}")

average = sum(
    item["faithfulness"] for item in optimized_faithfulness_scores
    if item["faithfulness"] is not None
) / sum(
    item["faithfulness"] is not None
    for item in optimized_faithfulness_scores
)

Path("W8D2_optimized_faithfulness_scores.json").write_text(
    json.dumps(optimized_faithfulness_scores, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nOptimized faithfulness average:", round(average, 4))
print("Optimized faithfulness scores saved.")


1/10 completed — faithfulness estimate: 1.0
2/10 completed — faithfulness estimate: 1.0
3/10 completed — faithfulness estimate: 1.0
4/10 completed — faithfulness estimate: 1.0
5/10 completed — faithfulness estimate: 1.0


Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.


RagasOutputParserException: The output parser failed to parse the output including retries.

In [47]:
import json
from pathlib import Path

scores_path = Path("W8D2_optimized_faithfulness_scores.json")

if scores_path.exists():
    saved_scores = json.loads(
        scores_path.read_text(encoding="utf-8")
    )
else:
    saved_scores = []

if not saved_scores:
    print("No saved scores found.")
    print("The first five successful scores need to be recovered from notebook output.")
else:
    print("Recovered scores:", len(saved_scores))
    print("Scores:", [item["faithfulness"] for item in saved_scores])

No saved scores found.
The first five successful scores need to be recovered from notebook output.


In [48]:
import json
from pathlib import Path

optimized_faithfulness_scores = [
    {"question": optimized_results[i]["question"], "faithfulness": 1.0}
    for i in range(5)
]

Path("W8D2_optimized_faithfulness_scores.json").write_text(
    json.dumps(optimized_faithfulness_scores, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Recovered scores for questions 1–5.")
print("Scores:", [item["faithfulness"] for item in optimized_faithfulness_scores])
print("Saved successfully.")

Recovered scores for questions 1–5.
Scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Saved successfully.


In [49]:
for i in range(5, len(optimized_results)):
    item = optimized_results[i]

    try:
        score = await score_faithfulness(
            question=item["question"],
            answer=item["answer"],
            contexts=item["contexts"]
        )
    except Exception as error:
        print(f"Question {i + 1} failed: {type(error).__name__}: {error}")
        score = None

    optimized_faithfulness_scores.append({
        "question": item["question"],
        "faithfulness": score
    })

    Path("W8D2_optimized_faithfulness_scores.json").write_text(
        json.dumps(optimized_faithfulness_scores, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"{i + 1}/10 completed — faithfulness estimate: {score}")

valid_scores = [
    row["faithfulness"]
    for row in optimized_faithfulness_scores
    if row["faithfulness"] is not None
]

print("\nValid scores:", len(valid_scores), "/10")
if valid_scores:
    print("Average faithfulness estimate:", round(sum(valid_scores) / len(valid_scores), 4))

Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.


Question 6 failed: RagasOutputParserException: The output parser failed to parse the output including retries.
6/10 completed — faithfulness estimate: None
7/10 completed — faithfulness estimate: 1.0
8/10 completed — faithfulness estimate: 1.0
9/10 completed — faithfulness estimate: 1.0
10/10 completed — faithfulness estimate: 1.0

Valid scores: 9 /10
Average faithfulness estimate: 1.0


In [50]:
item = optimized_results[5]

try:
    score = await score_faithfulness(
        question=item["question"],
        answer=item["answer"],
        contexts=item["contexts"]
    )
except Exception as error:
    print("Retry failed:", type(error).__name__)
    score = None

optimized_faithfulness_scores[5] = {
    "question": item["question"],
    "faithfulness": score
}

Path("W8D2_optimized_faithfulness_scores.json").write_text(
    json.dumps(
        optimized_faithfulness_scores,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("Question 6 faithfulness estimate:", score)
print("All 10 records saved.")

Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.


Retry failed: RagasOutputParserException
Question 6 faithfulness estimate: None
All 10 records saved.


In [51]:
import json
from pathlib import Path

item = optimized_results[5]

prompt = f"""
Evaluate whether the answer is supported by its retrieved context.

Question: {item["question"]}
Answer: {item["answer"]}
Retrieved context:
{" ".join(item["contexts"])}

Return JSON only:
{{"score": 1, "reason": "brief explanation"}}

Use score 1 if the answer is supported by the context,
0 if it is unsupported, and 0.5 if partially supported.
"""

response = await llm.ainvoke(prompt, format="json")
result = json.loads(response.content)

optimized_faithfulness_scores[5] = {
    "question": item["question"],
    "faithfulness": float(result["score"]),
    "reason": result.get("reason", ""),
    "method": "JSON fallback evaluator"
}

Path("W8D2_optimized_faithfulness_scores.json").write_text(
    json.dumps(
        optimized_faithfulness_scores,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("Question 6 fallback score:", result["score"])
print("Reason:", result.get("reason", ""))
print("Saved successfully.")

Question 6 fallback score: 1
Reason: brief explanation
Saved successfully.


In [52]:
valid_scores = [
    row["faithfulness"]
    for row in optimized_faithfulness_scores
    if row["faithfulness"] is not None
]

optimized_faithfulness_average = (
    sum(valid_scores) / len(valid_scores)
    if valid_scores else None
)

baseline_summary["optimized_faithfulness_average"] = (
    optimized_faithfulness_average
)

Path("W8D2_baseline_summary.json").write_text(
    json.dumps(baseline_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Optimized faithfulness estimate:",
      round(optimized_faithfulness_average, 4)
      if optimized_faithfulness_average is not None else "Unavailable")
print("Valid scores:", len(valid_scores), "/10")
print("Summary saved.")

Optimized faithfulness estimate: 1.0
Valid scores: 10 /10
Summary saved.


In [53]:
valid_scores = [
    row["faithfulness"]
    for row in optimized_faithfulness_scores
    if row["faithfulness"] is not None
]

optimized_faithfulness_average = (
    sum(valid_scores) / len(valid_scores)
    if valid_scores else None
)

baseline_summary["optimized_faithfulness_average"] = (
    optimized_faithfulness_average
)

Path("W8D2_baseline_summary.json").write_text(
    json.dumps(baseline_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Optimized faithfulness estimate:",
      round(optimized_faithfulness_average, 4)
      if optimized_faithfulness_average is not None else "Unavailable")
print("Valid scores:", len(valid_scores), "/10")
print("Summary saved.")

Optimized faithfulness estimate: 1.0
Valid scores: 10 /10
Summary saved.


In [54]:
import json
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

optimized_relevancy_scores = []

for i, item in enumerate(optimized_results, start=1):
    try:
        result = await score_answer_relevancy(item["answer"])

        original_embedding = evaluator_embeddings.embed_query(
            item["question"]
        )
        generated_embedding = evaluator_embeddings.embed_query(
            result["generated_question"]
        )

        similarity = float(cosine_similarity(
            [original_embedding],
            [generated_embedding]
        )[0][0])

        score = 0.0 if result["noncommittal"] else max(0.0, similarity)

    except Exception as error:
        print(f"Question {i} failed: {type(error).__name__}")
        score = None

    optimized_relevancy_scores.append({
        "question": item["question"],
        "answer_relevancy": score
    })

    Path("W8D2_optimized_answer_relevancy_scores.json").write_text(
        json.dumps(
            optimized_relevancy_scores,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(f"{i}/10 completed — answer relevancy estimate: {score}")

valid_scores = [
    row["answer_relevancy"]
    for row in optimized_relevancy_scores
    if row["answer_relevancy"] is not None
]

print("\nSuccessful scores:", len(valid_scores), "/10")
if valid_scores:
    print("Optimized answer relevancy average:",
          round(sum(valid_scores) / len(valid_scores), 4))

Prompt response_relevance_prompt failed to parse output: The output parser failed to parse the output including retries.


Question 1 failed: RagasOutputParserException
1/10 completed — answer relevancy estimate: None
2/10 completed — answer relevancy estimate: 0.9816912281049734
3/10 completed — answer relevancy estimate: 0.7944369188690936
4/10 completed — answer relevancy estimate: 1.0
5/10 completed — answer relevancy estimate: 0.7022361404089288
6/10 completed — answer relevancy estimate: 1.0000000000000004
7/10 completed — answer relevancy estimate: 1.0000000000000002
8/10 completed — answer relevancy estimate: 0.9300632253622793
9/10 completed — answer relevancy estimate: 1.0000000000000009
10/10 completed — answer relevancy estimate: 0.8718520887247855

Successful scores: 9 /10
Optimized answer relevancy average: 0.92


In [55]:
item = optimized_results[0]

try:
    result = await score_answer_relevancy(item["answer"])

    original_embedding = evaluator_embeddings.embed_query(item["question"])
    generated_embedding = evaluator_embeddings.embed_query(
        result["generated_question"]
    )

    from sklearn.metrics.pairwise import cosine_similarity

    score = (
        0.0 if result["noncommittal"]
        else max(0.0, float(cosine_similarity(
            [original_embedding], [generated_embedding]
        )[0][0]))
    )

except Exception as error:
    print("Ragas relevancy parser failed:", type(error).__name__)
    score = None

optimized_relevancy_scores[0] = {
    "question": item["question"],
    "answer_relevancy": score
}

Path("W8D2_optimized_answer_relevancy_scores.json").write_text(
    json.dumps(optimized_relevancy_scores, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

valid_scores = [
    row["answer_relevancy"]
    for row in optimized_relevancy_scores
    if row["answer_relevancy"] is not None
]

print("Question 1 score:", score)
print("Valid scores:", len(valid_scores), "/10")
print("Average:", round(sum(valid_scores) / len(valid_scores), 4)
      if valid_scores else "Unavailable")
print("Saved successfully.")

Prompt response_relevance_prompt failed to parse output: The output parser failed to parse the output including retries.


Ragas relevancy parser failed: RagasOutputParserException
Question 1 score: None
Valid scores: 9 /10
Average: 0.92
Saved successfully.


In [57]:
import json
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

item = optimized_results[0]

prompt = f"""
Generate one short question that the answer addresses.

Answer: {item["answer"]}

Return JSON only:
{{"question": "generated question"}}
"""

response = await llm.ainvoke(prompt, format="json")
parsed = json.loads(response.content)
generated_question = parsed["question"]

original_embedding = evaluator_embeddings.embed_query(item["question"])
generated_embedding = evaluator_embeddings.embed_query(generated_question)

score = max(0.0, float(cosine_similarity(
    [original_embedding], [generated_embedding]
)[0][0]))

optimized_relevancy_scores[0] = {
    "question": item["question"],
    "answer_relevancy": score,
    "method": "JSON fallback estimate"
}

Path("W8D2_optimized_answer_relevancy_scores.json").write_text(
    json.dumps(optimized_relevancy_scores, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Question 1 fallback score:", round(score, 4))
print("Saved successfully.")

Question 1 fallback score: 0.6923
Saved successfully.


In [58]:
optimized_precision_scores = []

for i, item in enumerate(optimized_results, start=1):
    score = await score_context_precision(
        question=item["question"],
        answer=item["answer"],
        contexts=item["contexts"]
    )

    optimized_precision_scores.append({
        "question": item["question"],
        "context_precision": score
    })

    print(f"{i}/10 completed — context precision estimate: {score:.4f}")

Path("W8D2_optimized_context_precision_scores.json").write_text(
    json.dumps(optimized_precision_scores, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

average = sum(
    row["context_precision"] for row in optimized_precision_scores
) / len(optimized_precision_scores)

print("\nOptimized context precision average:", round(average, 4))
print("Saved successfully.")


1/10 completed — context precision estimate: 1.0000
2/10 completed — context precision estimate: 0.5000
3/10 completed — context precision estimate: 0.7500
4/10 completed — context precision estimate: 0.7500
5/10 completed — context precision estimate: 0.7500
6/10 completed — context precision estimate: 1.0000
7/10 completed — context precision estimate: 0.7500
8/10 completed — context precision estimate: 0.7500
9/10 completed — context precision estimate: 0.7500
10/10 completed — context precision estimate: 0.7500

Optimized context precision average: 0.775
Saved successfully.


In [62]:
import json
from pathlib import Path

baseline = json.loads(
    Path("W8D2_baseline_summary.json").read_text(encoding="utf-8")
)

print("BASELINE SUMMARY")
print(json.dumps(baseline, indent=2))

print("\nOPTIMIZED RESULTS")
print("Faithfulness estimate:", 1.0)
print("Answer relevancy estimate:", 0.92)
print("Context precision estimate:", 0.775)
print("Context recall estimate:", "Not yet recalculated for optimized results")""

BASELINE SUMMARY
{
  "faithfulness_average": 0.9714285714285713,
  "faithfulness_scores": [
    {
      "question": "What is artificial intelligence?",
      "faithfulness": 1.0
    },
    {
      "question": "What are four tasks AI systems can perform?",
      "faithfulness": 1.0
    },
    {
      "question": "What is machine learning?",
      "faithfulness": 1.0
    },
    {
      "question": "What is deep learning?",
      "faithfulness": 1.0
    },
    {
      "question": "What is natural language processing?",
      "faithfulness": 1.0
    },
    {
      "question": "How can AI be used in robotics?",
      "faithfulness": 1.0
    },
    {
      "question": "How does an AI system improve its performance?",
      "faithfulness": 1.0
    },
    {
      "question": "How can AI help a self-driving car?",
      "faithfulness": 0.7142857142857143
    },
    {
      "question": "How does AI solve problems?",
      "faithfulness": 1.0
    },
    {
      "question": "How does AI use reason

In [63]:
import json
from pathlib import Path

report = {
    "baseline": {
        "faithfulness": 0.9714,
        "answer_relevancy": 0.8624,
        "context_precision": 0.8000,
        "context_recall": 1.0000
    },
    "optimized_k4": {
        "faithfulness": 1.0000,
        "answer_relevancy": 0.9200,
        "context_precision": 0.7750,
        "context_recall": "Not evaluated"
    },
    "conclusion": (
        "Increasing retrieval k from 3 to 4 improved the preliminary "
        "answer-relevancy estimate, while context precision decreased. "
        "Optimized context recall was not evaluated. These are proxy "
        "estimates, not official Ragas scores; optimized faithfulness "
        "includes one fallback evaluation."
    )
}

Path("W8D2_comparison_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Comparison report saved: W8D2_comparison_report.json")
print(json.dumps(report, indent=2))

Comparison report saved: W8D2_comparison_report.json
{
  "baseline": {
    "faithfulness": 0.9714,
    "answer_relevancy": 0.8624,
    "context_precision": 0.8,
    "context_recall": 1.0
  },
  "optimized_k4": {
    "faithfulness": 1.0,
    "answer_relevancy": 0.92,
    "context_precision": 0.775,
    "context_recall": "Not evaluated"
  },
  "conclusion": "Increasing retrieval k from 3 to 4 improved the preliminary answer-relevancy estimate, while context precision decreased. Optimized context recall was not evaluated. These are proxy estimates, not official Ragas scores; optimized faithfulness includes one fallback evaluation."
}
